# Step 4 — Production label extraction (REVISED)

**Settings: `GPU T4 x2`, `Internet On`.** Attach the competition data.

## How to run this

Run cells **1–5** interactively (setup → model → prompts → engine → smoke test). Takes ~8 minutes.

Then, once the smoke test passes, do **Save Version → Save & Run All** and close the tab. The full extraction takes ~2.5 hours and a committed run executes detached: it can't idle out, and it saves its outputs automatically. Come back later and collect the results.

Do *not* run the long cell interactively and then commit — that runs everything twice.

## What went wrong last time, and what's fixed

| problem | fix |
|---|---|
| one 3-hour cell, no checkpointing — a crash lost everything | checkpoints to disk every 10 batches, resumes automatically |
| fixed batch of 24 OOM'd on the long reports at the end | batches sized by **total token budget**, so long reports batch smaller |
| an OOM killed the whole run | OOM is caught, batch splits in half and retries, falls back to single reports |
| no way to test before committing 2.5 hours | smoke test cell on 48 reports first |

### 1 · Setup

In [ ]:
import os, re, gc, json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score

ROOT   = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train  = pd.read_csv(f"{ROOT}/train.csv")
LABELS = pd.read_csv(f"{ROOT}/sample_submission.csv").columns[1:].tolist()

KEYMAP = {"acl":"ACL","mcl":"MCL","medial_meniscus":"Medial Meniscus",
          "lateral_meniscus":"Lateral Meniscus","medial_oa":"Medial OA",
          "lateral_oa":"Lateral OA","pf_oa":"PF OA","effusion":"Effusion",
          "synovitis":"Synovitis","bakers":"Baker's","contusion":"Contusion",
          "fracture":"Fracture"}
KEYS = list(KEYMAP)

is_gold  = train[LABELS].notna().sum(axis=1) == len(LABELS)
reports  = train["Report"].fillna("").astype(str).tolist()
print(f"studies {len(train):,} | gold {is_gold.sum()} | reports {len(reports):,}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "*** NONE - set Accelerator ***")

### 2 · Model — 4-bit on a single GPU

One card with 4-bit weights beats sharding across two T4s: `device_map="auto"` splits by layer, which serialises the forward pass so one GPU idles while the other works.

In [ ]:
!pip install -q -U bitsandbytes 2>/dev/null

MODEL = "Qwen/Qwen2.5-7B-Instruct"
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"                       # required for batched decoder-only generation
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

t0 = time.time()
try:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                                 device_map={"": 0}, dtype=torch.float16)
    print("loaded 4-bit on cuda:0")
except Exception as e:
    print("4-bit unavailable, using fp16 sharded (slower):", e)
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="auto")
model.eval()
print(f"loaded in {time.time()-t0:.0f}s")

### 3 · Prompts

Two prompts, because the pilot showed they fail in opposite directions. **Categorical wins on ligaments** (ACL 0.835 vs 0.619) — a ligament is torn or it isn't, and grading blurs that. **Graded wins on osteoarthritis and effusion** (PF OA 0.770 vs 0.611) — those genuinely come in degrees.

In [ ]:
SYS_CAT = """You are a musculoskeletal radiologist extracting structured findings from knee MRI reports.
The report may be in ANY language. Read it in its original language.

For each finding, decide: PRESENT, ABSENT (negated/normal/intact), or NOT_MENTIONED.

- acl: anterior cruciate ligament tear or injury
- mcl: medial collateral ligament tear or injury
- medial_meniscus: medial meniscus tear
- lateral_meniscus: lateral meniscus tear
- medial_oa: MEDIAL tibiofemoral osteoarthritis / chondral loss
- lateral_oa: LATERAL tibiofemoral osteoarthritis / chondral loss
- pf_oa: patellofemoral osteoarthritis / chondromalacia
- effusion: joint effusion
- synovitis: synovial thickening or inflammation
- bakers: Baker's / popliteal cyst
- contusion: bone contusion, bruise, marrow oedema from trauma
- fracture: fracture of any bone

RULES:
1. Negation decides. "ACL intact", "menisci normal", "no tear" => absent.
2. Degenerative signal that is NOT a tear => absent for meniscus/ligament.
3. Distinguish MEDIAL from LATERAL carefully.
4. Never guess: if not referred to, say not_mentioned.

Reply ONLY JSON:
{"acl":"present|absent|not_mentioned","mcl":"...","medial_meniscus":"...","lateral_meniscus":"...","medial_oa":"...","lateral_oa":"...","pf_oa":"...","effusion":"...","synovitis":"...","bakers":"...","contusion":"...","fracture":"..."}"""


SYS_NUM = """You are a musculoskeletal radiologist reading knee MRI reports and estimating how likely each of twelve findings is to be truly present.

The report may be in ANY language. Read it in its original language.

For EACH finding give an integer 0-100: the likelihood an expert reviewing the actual images would mark it present.

  0-10    explicitly negated ("ACL intact", "menisci normal", "no effusion")
  11-30   not mentioned at all
  31-55   equivocal, borderline, trace, minimal, degenerative change short of a tear
  56-80   clearly described but mild, or described indirectly
  81-100  explicitly and unambiguously present

THE TWELVE FINDINGS (match on MEANING, not exact wording):
acl              ACL tear, rupture, discontinuity
mcl              MCL tear, rupture, high-grade sprain
medial_meniscus  medial meniscus TEAR (degeneration alone is NOT a tear -> low)
lateral_meniscus lateral meniscus TEAR (same rule)
medial_oa        MEDIAL compartment: osteoarthritis, chondral loss/thinning, joint space
                 narrowing, subchondral sclerosis, osteophytes
lateral_oa       LATERAL compartment: same features
pf_oa            PATELLOFEMORAL (patella/trochlea): same features, chondromalacia patellae,
                 retropatellar cartilage thinning
effusion         joint effusion, intra-articular fluid (trace/small counts, ~50-65)
synovitis        synovial thickening, enhancement, inflammation, plica synovitis
bakers           Baker's cyst, popliteal cyst, gastrocnemio-semimembranosus bursa cyst
contusion        bone contusion, bruise, marrow oedema from trauma
fracture         fracture, cortical break, avulsion, insufficiency/stress fracture

RULES:
- Negation is decisive: "intact", "normal", "unremarkable", "no evidence of" => under 10.
- "Tricompartmental osteoarthritis" raises ALL THREE OA findings.
- Meniscal degeneration without a tear line => 20-40.
- If never referred to, give 15-25. Never exactly 0 for silence.

Reply ONLY JSON of twelve integers:
{"acl":0,"mcl":0,"medial_meniscus":0,"lateral_meniscus":0,"medial_oa":0,"lateral_oa":0,"pf_oa":0,"effusion":0,"synovitis":0,"bakers":0,"contusion":0,"fracture":0}"""

print("prompts ready")

### 4 · Extraction engine

Three things this gets right that the last version didn't:

- **Token-budget batching.** Reports are sorted by length and grouped so that `longest_in_batch x batch_size` stays under a cap. Short reports batch wide, long reports batch narrow. This is what prevents the OOM that hit at report 3,856 last time.
- **Checkpointing.** Writes to JSON every 10 batches and resumes from it on re-run.
- **OOM recovery.** Catches the error, halves the batch, retries. Only gives up on a single report that won't fit alone.

In [ ]:
def run_extract(reports, system, ckpt, max_new_tokens=90, tok_budget=18000, tag=""):
    out = [None] * len(reports)
    if os.path.exists(ckpt):
        for i, v in json.load(open(ckpt)).items():
            out[int(i)] = v
        print(f"  [{tag}] resumed {sum(x is not None for x in out)}/{len(out)}")

    todo = [i for i, x in enumerate(out) if x is None]
    if not todo:
        print(f"  [{tag}] already complete")
        return [x if x is not None else {} for x in out]
    todo.sort(key=lambda i: len(reports[i]))

    SYS_TOK = len(tok(system).input_ids)
    batches, cur, cur_max = [], [], 0
    for i in todo:
        est = min(len(reports[i]) // 3 + SYS_TOK + 60, 3072)
        if cur and max(cur_max, est) * (len(cur) + 1) > tok_budget:
            batches.append(cur); cur, cur_max = [i], est
        else:
            cur.append(i); cur_max = max(cur_max, est)
    if cur: batches.append(cur)
    print(f"  [{tag}] {len(todo)} reports -> {len(batches)} batches")

    def save():
        json.dump({str(i): v for i, v in enumerate(out) if v is not None}, open(ckpt, "w"))

    t0, done0, n, bn = time.time(), len(reports) - len(todo), 0, 0
    while bn < len(batches):
        idx = batches[bn]
        while True:
            try:
                texts = [tok.apply_chat_template(
                            [{"role": "system", "content": system},
                             {"role": "user",   "content": f"REPORT:\n{reports[i][:6000]}"}],
                            tokenize=False, add_generation_prompt=True) for i in idx]
                enc = tok(texts, return_tensors="pt", padding=True,
                          truncation=True, max_length=3072).to(model.device)
                with torch.no_grad():
                    gen = model.generate(**enc, max_new_tokens=max_new_tokens,
                                         do_sample=False, temperature=None, top_p=None,
                                         top_k=None, pad_token_id=tok.pad_token_id)
                new = gen[:, enc.input_ids.shape[1]:]
                for j, i in enumerate(idx):
                    txt = tok.decode(new[j], skip_special_tokens=True)
                    m = re.search(r"\{.*?\}", txt, re.S)
                    try:    out[i] = json.loads(m.group(0)) if m else {}
                    except Exception: out[i] = {}
                del enc, gen, new
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); gc.collect()
                if len(idx) == 1:
                    out[idx[0]] = {}
                    break
                half = len(idx) // 2
                print(f"  [{tag}] OOM -> splitting {len(idx)} into {half}+{len(idx)-half}")
                batches.insert(bn + 1, idx[half:])
                idx = idx[:half]

        n += len(idx); bn += 1
        if bn % 10 == 0 or bn == len(batches):
            save()
            el = time.time() - t0
            print(f"  [{tag}] {done0+n}/{len(reports)}  {el/max(n,1):.2f}s/rep  eta {(len(todo)-n)*el/max(n,1)/60:.0f} min")
    save()
    return [x if x is not None else {} for x in out]

print("engine ready")

### 5 · Smoke test — run this before committing

48 reports, ~1 minute. Deliberately takes the **longest** reports, since those are what broke the previous run. If this passes, the full job will too.

In [ ]:
longest = sorted(range(len(reports)), key=lambda i: -len(reports[i]))[:48]
sample  = [reports[i] for i in longest]

t = run_extract(sample, SYS_NUM, "/kaggle/working/ckpt_smoke.json", tag="smoke")
ok = sum(1 for x in t if x and len(x) >= 10)
print(f"\nparsed {ok}/48")
print("example:", t[0])
os.remove("/kaggle/working/ckpt_smoke.json")

assert ok >= 44, "extraction unreliable - stop and investigate before committing"
print("\nSMOKE TEST PASSED -> Save Version -> Save & Run All, then close the tab")

### 6 · Full extraction — ~2.5 hours

Runs on commit. Two sequential passes with independent checkpoints, so if the second fails the first is still safe.

In [ ]:
print(f"extracting over {len(reports):,} reports\n")
print("PASS 1/2 - categorical")
cat  = run_extract(reports, SYS_CAT, "/kaggle/working/ckpt_cat.json", tag="cat")
print("\nPASS 2/2 - graded")
numr = run_extract(reports, SYS_NUM, "/kaggle/working/ckpt_num.json", tag="num")
print(f"\nparse failures: cat={sum(1 for x in cat if not x)}  num={sum(1 for x in numr if not x)}")

### 7 · Blend and validate

Weights come from the pilot's per-label AUCs: ligaments lean categorical, osteoarthritis and effusion lean graded, the rest split evenly. Validated on the 58 gold studies — the same set used to choose the weights, so read the number as indicative, not exact.

In [ ]:
CAT2NUM = {"present": 90.0, "not_mentioned": 20.0, "absent": 5.0}
W = {"acl":0.80, "mcl":0.80,
     "medial_meniscus":0.35, "lateral_meniscus":0.35,
     "medial_oa":0.20, "lateral_oa":0.20, "pf_oa":0.20,
     "effusion":0.20, "synovitis":0.35,
     "bakers":0.50, "contusion":0.50, "fracture":0.50}

def cnum(d, k): return CAT2NUM.get(str(d.get(k, "not_mentioned")).lower(), 20.0)
def nnum(d, k):
    try:    return min(max(float(d.get(k, 20)), 0), 100)
    except Exception: return 20.0

blend = pd.DataFrame({k: [W[k]*cnum(c,k) + (1-W[k])*nnum(n,k) for c, n in zip(cat, numr)]
                      for k in KEYS})
blend.insert(0, "StudyInstanceUID", train["StudyInstanceUID"].values)

g = is_gold.values
print(f"{'label':<18}{'cat':>8}{'num':>8}{'blend':>8}{'n_pos':>7}")
ac, an, ab = [], [], []
for k in KEYS:
    y = pd.to_numeric(train.loc[g, KEYMAP[k]], errors="coerce").fillna(0).astype(int).values
    if y.min() == y.max(): continue
    sc = np.array([cnum(cat[i], k)  for i in np.where(g)[0]])
    sn = np.array([nnum(numr[i], k) for i in np.where(g)[0]])
    sb = blend.loc[g, k].values
    a1, a2, a3 = roc_auc_score(y, sc), roc_auc_score(y, sn), roc_auc_score(y, sb)
    ac.append(a1); an.append(a2); ab.append(a3)
    print(f"{KEYMAP[k]:<18}{a1:>8.3f}{a2:>8.3f}{a3:>8.3f}{y.sum():>7}")
print(f"\nMACRO  cat={np.mean(ac):.3f}  num={np.mean(an):.3f}  BLEND={np.mean(ab):.3f}")
print("pilot reference: cat 0.770 | num 0.794 | per-label best 0.831")

### 8 · Save

`pseudo_labels.csv` holds continuous 0–100 scores. **Train on these as soft targets, not thresholded 0/1** — the score carries the extractor's confidence, so an ambiguous report teaches the model less than a clear one. Since the metric is AUC, which only cares about ranking, you never need a threshold anywhere in this pipeline.

`gold58.csv` stays separate. Never train on it. It's the only unbiased signal you have.

In [ ]:
blend.to_csv("pseudo_labels.csv", index=False)
train.loc[g, ["StudyInstanceUID"] + LABELS].to_csv("gold58.csv", index=False)
pd.DataFrame(cat ).assign(StudyInstanceUID=train.StudyInstanceUID.values).to_csv("raw_categorical.csv", index=False)
pd.DataFrame(numr).assign(StudyInstanceUID=train.StudyInstanceUID.values).to_csv("raw_graded.csv", index=False)
print("wrote pseudo_labels.csv, gold58.csv, raw_categorical.csv, raw_graded.csv\n")

print("implied prevalence at score>50 vs gold58 (gold58 is an enriched sample, so gaps are expected):")
for k in KEYS:
    gp = pd.to_numeric(train.loc[g, KEYMAP[k]], errors="coerce").fillna(0).mean()
    print(f"  {KEYMAP[k]:<18} pseudo {(blend[k] > 50).mean():>6.1%}   gold58 {gp:>6.1%}")